# Design > Design of Experiments (DOE)

<div class="alert alert-info">Create (partial) factorial experimental designs with D-optimal efficiency</div>

The `doe` function creates factorial experimental designs. A **full factorial** design includes every possible combination of factor levels. When the full factorial is too large to run, a **partial factorial** design selects a subset of profiles that maximizes D-efficiency while maintaining balance.

The `doe` function uses a Federov exchange algorithm to search for D-optimal designs. When the number of trials is divisible by all factor level counts, the algorithm enforces **balance** (each level appears equally often).

Factor specifications can be provided as a dictionary (recommended) or as a semicolon-delimited string. Storing factors in JSON files makes them easy to version-control and share.


In [2]:
import json

import pyrsm as rsm

In [3]:
## setup pyrsm for autoreload
%reload_ext autoreload
%autoreload 2
%aimport pyrsm

# Example 1: Movie theater conjoint (3 factors)

A movie theater chain wants to understand customer preferences for three attributes:

- **Price**: $10, $13, $16
- **Sight**: staggered seating or not staggered
- **Food**: hotdogs and popcorn, gourmet food, or no food

The full factorial has 3 x 2 x 3 = **18** profiles. We load the factor definitions from a JSON file and explore which partial factorials are efficient.


In [4]:
with open("../data/design/factors-movie.json") as f:
    factors_movie = json.load(f)

factors_movie

{'price': ['$10', '$13', '$16'],
 'sight': ['staggered', 'not staggered'],
 'food': ['hotdogs and popcorn', 'gourmet food', 'no food']}

In [5]:
d = rsm.design.doe(factors_movie, seed=1234)
d.summary(full=False)

Experimental design
# trials for partial factorial: 18
# trials for full factorial   : 18
Random seed                   : 1234

Attributes and levels:
price: $10, $13, $16
sight: staggered, not staggered
food: hotdogs and popcorn, gourmet food, no food

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 6      ┆ 0.513        ┆ true     │
│ 7      ┆ 0.408        ┆ false    │
│ 8      ┆ 0.334        ┆ false    │
│ 9      ┆ 0.846        ┆ false    │
│ 10     ┆ 0.752        ┆ false    │
│ 11     ┆ 0.669        ┆ false    │
│ 12     ┆ 0.875        ┆ true     │
│ 13     ┆ 0.798        ┆ false    │
│ 14     ┆ 0.805        ┆ false    │
│ 15     ┆ 0.779        ┆ false    │
│ 16     ┆ 0.717        ┆ false    │
│ 17     ┆ 0.659        ┆ false    │
│ 18     ┆ 1.000        ┆ true     │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌───────┬────────┬────────┬────────┐
│       

## Understanding the efficiency table

The efficiency table shows D-efficiency and balance status for each possible trial count, starting from the minimum (= total number of levels - number of factors + 1). The algorithm stops when it finds a design with D-efficiency of 1.0 (which is the full factorial or an equally efficient subset).

**Rule of thumb for selecting a partial factorial:**

1. **D-efficiency > 0.8** - Ensures the design provides reasonably precise estimates of main effects
2. **Balanced** - Each level of each factor appears the same number of times

Look at the efficiency table above and find the smallest trial count that satisfies both criteria.


## Selecting a partial factorial

From the efficiency table, we can select a specific number of trials. For example, with 6 trials:


In [ ]:
d = rsm.design.doe(factors_movie)
d.summary()

Experimental design
# trials for partial factorial: 18
# trials for full factorial   : 18

Attributes and levels:
price: $10, $13, $16
sight: staggered, not staggered
food: hotdogs and popcorn, gourmet food, no food

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 6      ┆ 0.513        ┆ true     │
│ 7      ┆ 0.408        ┆ false    │
│ 8      ┆ 0.334        ┆ false    │
│ 9      ┆ 0.846        ┆ false    │
│ 10     ┆ 0.752        ┆ false    │
│ 11     ┆ 0.669        ┆ false    │
│ 12     ┆ 0.875        ┆ true     │
│ 13     ┆ 0.798        ┆ false    │
│ 14     ┆ 0.805        ┆ false    │
│ 15     ┆ 0.779        ┆ false    │
│ 16     ┆ 0.717        ┆ false    │
│ 17     ┆ 0.659        ┆ false    │
│ 18     ┆ 1.000        ┆ true     │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌───────┬────────┬────────┬────────┐
│       ┆ price  ┆ sight  ┆ food   │
╞═══════

In [9]:
d12 = rsm.design.doe(factors_movie, trials=12)
d12.part

trial,price,sight,food
i64,enum,enum,enum
1,"""$10""","""staggered""","""hotdogs and popcorn"""
2,"""$10""","""staggered""","""gourmet food"""
5,"""$10""","""not staggered""","""gourmet food"""
6,"""$10""","""not staggered""","""no food"""
7,"""$13""","""staggered""","""hotdogs and popcorn"""
9,"""$13""","""staggered""","""no food"""
10,"""$13""","""not staggered""","""hotdogs and popcorn"""
11,"""$13""","""not staggered""","""gourmet food"""
14,"""$16""","""staggered""","""gourmet food"""


In [6]:
d6.part

trial,price,sight,food
i64,enum,enum,enum
1,"""$10""","""staggered""","""hotdogs and popcorn"""
6,"""$10""","""not staggered""","""no food"""
8,"""$13""","""staggered""","""gourmet food"""
10,"""$13""","""not staggered""","""hotdogs and popcorn"""
15,"""$16""","""staggered""","""no food"""
17,"""$16""","""not staggered""","""gourmet food"""


<div class="alert alert-warning">

**Important: The no-interaction assumption**

When you use a partial factorial design, you are implicitly assuming that **interactions between factors are negligible**. The smallest partial factorials can estimate main effects but **cannot** accurately capture interaction effects. Before choosing a partial factorial, ask yourself: _Am I comfortable assuming that the effect of one factor does not depend on the level of another factor?_

For example, if customers' sensitivity to price depends on the type of food available, then there is a price-food interaction that a small partial factorial cannot estimate.

If interactions are likely important, you should use a larger partial factorial or the full factorial.

</div>


In [7]:
print("Estimable effects with 6 trials:")
for e in d6.estimable():
    print(f"  {e}")

Estimable effects with 6 trials:
  price|$13
  price|$16
  sight|not staggered
  food|gourmet food
  food|no food


In [8]:
# Compare to full factorial
d_full = rsm.design.doe(factors_movie)
print("Estimable effects with full factorial (18 trials):")
for e in d_full.estimable():
    print(f"  {e}")

Estimable effects with full factorial (18 trials):
  price|$13
  price|$16
  sight|not staggered
  food|gourmet food
  food|no food
  price|$13:sight|not staggered
  price|$16:sight|not staggered
  price|$13:food|gourmet food
  price|$13:food|no food
  price|$16:food|gourmet food
  price|$16:food|no food
  sight|not staggered:food|gourmet food
  sight|not staggered:food|no food


# Example 2: Online tools promotion (2 factors)

An online retailer wants to test shipping and discount offers:

- **Free shipping threshold**: $300 or $200
- **Discount**: 15% or 20%

With only 2 x 2 = **4** total profiles, the full factorial is small enough to run:


In [9]:
with open("../data/design/factors-tools.json") as f:
    factors_tools = json.load(f)

d_tools = rsm.design.doe(factors_tools)
d_tools.summary()

Experimental design
# trials for partial factorial: 4
# trials for full factorial   : 4

Attributes and levels:
free_ship: $300, $200
discount: 15%, 20%

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 3      ┆ 0.135        ┆ false    │
│ 4      ┆ 1.000        ┆ true     │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌───────────┬───────────┬──────────┐
│           ┆ free_ship ┆ discount │
╞═══════════╪═══════════╪══════════╡
│ free_ship ┆ 1.000     ┆ -0.000   │
│ discount  ┆ -0.000    ┆ 1.000    │
└───────────┴───────────┴──────────┘

Partial factorial design:
┌───────┬───────────┬──────────┐
│ trial ┆ free_ship ┆ discount │
╞═══════╪═══════════╪══════════╡
│ 1     ┆ $300      ┆ 15%      │
│ 2     ┆ $300      ┆ 20%      │
│ 3     ┆ $200      ┆ 15%      │
│ 4     ┆ $200      ┆ 20%      │
└───────┴───────────┴──────────┘

Full factorial design:
┌───────┬──────

# Example 3: Online tools with coupon entry (3 factors)

Adding a third factor for coupon entry method:

- **Free shipping threshold**: $300 or $200
- **Discount**: 15% or 20%
- **Coupon entry**: manual or automatic

The full factorial has 2 x 2 x 2 = **8** profiles.


In [10]:
with open("../data/design/factors-tools-ship-discount.json") as f:
    factors_tools3 = json.load(f)

d_tools3 = rsm.design.doe(factors_tools3, seed=1234)
d_tools3.summary(full=False)

Experimental design
# trials for partial factorial: 4
# trials for full factorial   : 8
Random seed                   : 1234

Attributes and levels:
free_ship: $300, $200
discount: 15%, 20%
coup_entry: manual, automatic

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 4      ┆ 1.000        ┆ true     │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌────────────┬───────────┬──────────┬────────────┐
│            ┆ free_ship ┆ discount ┆ coup_entry │
╞════════════╪═══════════╪══════════╪════════════╡
│ free_ship  ┆ 1.000     ┆ -0.000   ┆ -0.000     │
│ discount   ┆ -0.000    ┆ 1.000    ┆ -0.000     │
│ coup_entry ┆ -0.000    ┆ -0.000   ┆ 1.000      │
└────────────┴───────────┴──────────┴────────────┘

Partial factorial design:
┌───────┬───────────┬──────────┬────────────┐
│ trial ┆ free_ship ┆ discount ┆ coup_entry │
╞═══════╪═══════════╪══════════╪════════════╡


# Example 4: MP3 player conjoint (5 factors)

A consumer electronics company wants to test MP3 player designs with 5 attributes:

- **Memory**: 4GB, 6GB, 8GB
- **FM radio**: Yes, No
- **Size**: Small, Medium, Large
- **Price**: $50, $100, $150
- **Shape**: Square, Circular, Rectangular

The full factorial has 3 x 2 x 3 x 3 x 3 = **162** profiles — far too many to test. We generate a partial factorial with 18 trials (the smallest balanced trial count divisible by all factor levels: lcm(3,2,3,3,3) = 6, and 18 = 6 x 3).


In [11]:
with open("../data/design/factors-mp3.json") as f:
    factors_mp3 = json.load(f)

d_mp3 = rsm.design.doe(factors_mp3, trials=18, seed=1234)
d_mp3.summary(full=False)

Experimental design
# trials for partial factorial: 18
# trials for full factorial   : 162
Random seed                   : 1234

Attributes and levels:
Memory: 4GB, 6GB, 8GB
FM_radio: Yes, No
Size: Small, Medium, Large
Price: $50, $100, $150
Shape: Square, Circular, Rectangular

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 18     ┆ 0.233        ┆ false    │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌──────────┬────────┬──────────┬───────┬───────┬───────┐
│          ┆ Memory ┆ FM_radio ┆ Size  ┆ Price ┆ Shape │
╞══════════╪════════╪══════════╪═══════╪═══════╪═══════╡
│ Memory   ┆ 1.000  ┆ 0.337    ┆ 0.311 ┆ 0.310 ┆ 0.282 │
│ FM_radio ┆ 0.337  ┆ 1.000    ┆ 0.141 ┆ 0.491 ┆ 0.587 │
│ Size     ┆ 0.311  ┆ 0.141    ┆ 1.000 ┆ 0.100 ┆ 0.486 │
│ Price    ┆ 0.310  ┆ 0.491    ┆ 0.100 ┆ 1.000 ┆ 0.202 │
│ Shape    ┆ 0.282  ┆ 0.587    ┆ 0.486 ┆ 0.202 ┆ 1.000 │
└──────

# Example 5: Harrah's casino promotion (6 factors)

A casino wants to test promotional offers with 6 attributes. With many factors, the full factorial becomes very large (3 x 3 x 2 x 3 x 2 x 2 = **216** profiles) and a partial factorial is essential.

- **Free nights**: 1, 2, 3
- **Free chips**: $100, $50, $0
- **Free shows**: yes, no
- **Expiration date**: 12 months, 3 months, 6 months
- **Phone follow-up**: no, yes
- **Email follow-up**: no, yes

We try 18 trials (divisible by 2 and 3, so balance is possible):


In [12]:
with open("../data/design/factors-harrahs.json") as f:
    factors_harrahs = json.load(f)

d_harrahs = rsm.design.doe(factors_harrahs, trials=18, seed=1234)
d_harrahs.summary(full=False)

Experimental design
# trials for partial factorial: 18
# trials for full factorial   : 216
Random seed                   : 1234

Attributes and levels:
free_nights: 1, 2, 3
free_chips: $100, $50, 0
free_shows: yes, no
exp_date: 12 months, 3 months, 6 months
phone_follow: no, yes
email_follow: no, yes

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 18     ┆ 0.129        ┆ false    │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌──────────────┬─────────────┬────────────┬────────────┬──────────┬──────────────┬──────────────┐
│              ┆ free_nights ┆ free_chips ┆ free_shows ┆ exp_date ┆ phone_follow ┆ email_follow │
╞══════════════╪═════════════╪════════════╪════════════╪══════════╪══════════════╪══════════════╡
│ free_nights  ┆ 1.000       ┆ -0.002     ┆ -0.110     ┆ 0.230    ┆ -0.099       ┆ -0.140       │
│ free_chips   ┆ -0.002      ┆ 1.000      ┆ 0.54

# Example 6: Full movie theater conjoint (5 factors)

Expanding the movie theater example to include all 5 attributes:

- **Price**: $10, $13, $16
- **Sight**: Staggered, Not Staggered
- **Comfort**: Average seat without cupholder, Average seat with cupholder, Large seat with cupholder
- **Audio/Visual**: Small screen with plain sound, Large screen with plain sound, Large screen with digital sound
- **Food**: No food, Hot dogs and popcorn, Gourmet food

The full factorial has 3 x 2 x 3 x 3 x 3 = **162** profiles. We try 18 trials:


In [13]:
with open("../data/design/factors-movie-full.json") as f:
    factors_movie_full = json.load(f)

d_movie_full = rsm.design.doe(factors_movie_full, trials=18, seed=1234)
d_movie_full.summary(full=False)

Experimental design
# trials for partial factorial: 18
# trials for full factorial   : 162
Random seed                   : 1234

Attributes and levels:
price: $10, $13, $16
sight: Staggered, Not Staggered
comfort: Average seat without cupholder, Average seat with cupholder, Large seat with cupholder
audio_visual: Small screen with plain sound, Large screen with plain sound, Large screen with digital sound
food: No food, Hot dogs and popcorn, Gourmet food

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 18     ┆ 0.233        ┆ false    │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌──────────────┬───────┬───────┬─────────┬──────────────┬───────┐
│              ┆ price ┆ sight ┆ comfort ┆ audio_visual ┆ food  │
╞══════════════╪═══════╪═══════╪═════════╪══════════════╪═══════╡
│ price        ┆ 1.000 ┆ 0.337 ┆ 0.311   ┆ 0.310        ┆ 0.282 │
│ sight        ┆ 0.

# Including interactions

If you suspect that certain factors interact (e.g., the effect of price depends on the food option), you can explicitly include interaction terms in the design. This requires more trials but allows you to estimate those interactions:


In [14]:
d_int = rsm.design.doe(
    factors_movie,
    interactions="price:food",
    trials=18,
    seed=1234,
)
print(f"D-efficiency: {d_int.Dea}")
print("\nEstimable effects:")
for e in d_int.estimable():
    print(f"  {e}")

D-efficiency: 1.0

Estimable effects:
  price|$13
  price|$16
  sight|not staggered
  food|gourmet food
  food|no food
  price|$13:sight|not staggered
  price|$16:sight|not staggered
  price|$13:food|gourmet food
  price|$13:food|no food
  price|$16:food|gourmet food
  price|$16:food|no food
  sight|not staggered:food|gourmet food
  sight|not staggered:food|no food


With the full factorial (18 trials), all main effects and interactions are estimable.


# Defining factors inline

You can also define factors directly as a Python dictionary without loading from a file:


In [15]:
factors_inline = {
    "color": ["red", "blue", "green"],
    "size": ["small", "large"],
}

d_inline = rsm.design.doe(factors_inline, seed=1234)
d_inline.summary(full=False, est=False)

Experimental design
# trials for partial factorial: 6
# trials for full factorial   : 6
Random seed                   : 1234

Attributes and levels:
color: red, blue, green
size: small, large

Design efficiency:
┌────────┬──────────────┬──────────┐
│ Trials ┆ D-efficiency ┆ Balanced │
╞════════╪══════════════╪══════════╡
│ 4      ┆ 0.135        ┆ false    │
│ 5      ┆ 0.223        ┆ false    │
│ 6      ┆ 1.000        ┆ true     │
└────────┴──────────────┴──────────┘

Partial factorial design correlations (polychoric):
┌───────┬───────┬───────┐
│       ┆ color ┆ size  │
╞═══════╪═══════╪═══════╡
│ color ┆ 1.000 ┆ 0.000 │
│ size  ┆ 0.000 ┆ 1.000 │
└───────┴───────┴───────┘

Partial factorial design:
┌───────┬───────┬───────┐
│ trial ┆ color ┆ size  │
╞═══════╪═══════╪═══════╡
│ 1     ┆ red   ┆ small │
│ 2     ┆ red   ┆ large │
│ 3     ┆ blue  ┆ small │
│ 4     ┆ blue  ┆ large │
│ 5     ┆ green ┆ small │
│ 6     ┆ green ┆ large │
└───────┴───────┴───────┘


# Summary: Choosing a partial factorial

When designing a conjoint study or experiment:

1. **Start with the full factorial** to understand the total number of profiles
2. **Review the efficiency table** to find the range of possible partial factorials
3. **Apply the rule of thumb**: select the smallest design with **D-efficiency > 0.8** and **Balanced = True**
4. **Consider interactions**: if you believe interactions between factors are important, you need more trials. The smallest partials estimate main effects only

The key trade-off is between the number of trials (cost/effort) and the ability to estimate effects. Smaller designs are cheaper but require the assumption that interactions are negligible.


© Vincent Nijs (2026)
